In [ ]:
#Import libraries

from datasets import load_dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

In [ ]:
RANDOM_SEED = 42

In [ ]:
#Load dataset
ds = load_dataset("Fllamber/TCGA-PANCAN-HiSeq-2770x20530")
print(ds)

In [ ]:
#dimensions
ds.shape

In [ ]:
#example of first row
print(ds['train'][0])

The values show the measured expression level of the gene in each sample.

In [ ]:
#check the unique cancer types
print(set(ds["train"]["cancer"]))

In [ ]:
#convert dataset to pandas datafraem
ds.set_format(type='pandas')
df = ds['train'][:]

print('Dataframe shape:', df.shape)
df.head()

The dataset contains 2770 samples and 20530 gene-expression featues with a 'cancer' target column.

In [ ]:
#Number of samples per cancer type for balanced or imbalanced dataset
counts_per_class = df['cancer'].value_counts()
print(counts_per_class)

In [ ]:
#visualize class distribution
plt.figure(figsize=(9,5))

counts_per_class.plot(kind='bar', color=['blue', 'green', 'red', 'orange', 'purple'])
plt.xlabel('Cancer type')
plt.ylabel("Number of samples per cancer type")
plt.title('Distribution of Cancer Types')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The training data has a class imbalance, with more BRCA samples than the other cancer types.

In [ ]:
#Check for missing values
print("Sum of missing values:", df.isnull().sum().sum())

In [ ]:
#Inspect data type and structure 
df.info()

In [ ]:
X = df.drop(columns=['cancer']) #features to make predicitions 
y = df['cancer'] #target learn to predict 

In [ ]:
#stratify ensures that the training and testing sets have the same proportion
#of classes as the original dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_SEED)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_SEED)

In [ ]:
print("Training:", X_fit.shape)
print("Validation:", X_val.shape)

In [ ]:
#Statistics
X.describe()

In [ ]:
#Standardization

#Fit scalar on fitting data 
scaler = StandardScaler()
scaler.fit(X_fit)
X_fit_scaled = scaler.transform(X_fit)

#apply fitted scaler on validation and test set
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Baseline logistic regression

#train baseline model
baseline_logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_SEED
)

baseline_logreg.fit(X_fit_scaled, y_fit)

#predict on scaled validation set
y_val_pred_baselinelogreg = baseline_logreg.predict(X_val_scaled)

#Evaluate baseline

print("Baseline accuracy:", round(
      accuracy_score(y_val, y_val_pred_baselinelogreg), 3))
print("Baseline Macro-F1:", round(
      f1_score(y_val, y_val_pred_baselinelogreg, 
               average="macro"), 3))
print("Classification report:")
print(classification_report(y_val, y_val_pred_baselinelogreg))

The baseline model achieved an accuracy of 98,45% and a macro-F1 score of 98,31%, It performed well across all the cancer types. The macro-F1 is used as the main evaliton metric since it gives equal imporantce to each cancer type.

In [ ]:
#Check best number of principal components

#compare different variance thresholds
variance_threshold = [0.80, 0.90, 0.95, 0.99]
results_pca = []

for variance in variance_threshold:
    pca_test = PCA(n_components=variance)

    X_fit_pca_test = pca_test.fit_transform(X_fit_scaled)

    X_val_pca_test = pca_test.transform(X_val_scaled)

    pca_model_log = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED
    )
    
    pca_model_log.fit(X_fit_pca_test, y_fit)

    y_val_pred_logreg = pca_model_log.predict(X_val_pca_test)

    pca_accuracy = accuracy_score(y_val, y_val_pred_logreg)
    pca_macro_f1 = f1_score(y_val, y_val_pred_logreg, average='macro')

    results_pca.append({
        "Variance threshold": variance,
        "Components": pca_test.n_components_,
        "Accuracy": pca_accuracy,
        "Macro f1-score": pca_macro_f1
    })

    print(f"PCA variance: {variance}")
    print(f"Number of components: {pca_test.n_components_}")
    print("Classification report:")
    print(classification_report(
        y_val, 
        y_val_pred_logreg))

results_pca_df = pd.DataFrame(results_pca)
print(results_pca_df.round(3))

PCA reduced the number of features largely while slightly improving the performancd compared to the baseline logistic regression model across all cancer types. PCA 80% reduces the data to 324 components and improves the Macro-F1 to 0.987, PCA 90% reduces the data to 597 components but produces the same scores as PCA 80%. PCA 95% uses 846 components and has a slighlty higher Macro-F1 of 0.988. PCA 99% uses 1258 components and has the highest accuracny and macro f1-score of 0.992. To choose between a right balance of dimensionality reduction and performance, the 95% variance threshold is selected. 

In [ ]:
#Inspect PCA 95% representation
pca = PCA(n_components=0.95)
X_fit_pca = pca.fit_transform(X_fit_scaled)
X_val_pca = pca.transform(X_val_scaled)

print('Variance retained:', round(pca.explained_variance_ratio_.sum(), 4))
print("\nOriginal shape:", X_fit_scaled.shape)
print("PCA shape:", X_fit_pca.shape)
print("Number of components:", pca.n_components_)
print("Validation PCA shape:", X_val_pca.shape)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print("Total variance retained:", cumulative_variance[-1])


plt.figure(figsize=(8,5))

plt.plot(range(1, len(cumulative_variance) +1), cumulative_variance)
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA: Cumulative explained variance")
plt.legend()
plt.grid(True)
plt.show()

PCA performance before t-SNE reduces computational cost.

In [ ]:
#Visualize training data with t-SNE

#reduce fitting data to 50 PCA components before t-SNE
pca_tsne = PCA(n_components=50, random_state=RANDOM_SEED)

X_fit_pca50 = pca_tsne.fit_transform(X_fit_scaled)

tsne_mod = TSNE(n_components=2,
            perplexity=30,
            init="pca",
            learning_rate="auto",
            random_state=RANDOM_SEED)

X_fit_tsne = tsne_mod.fit_transform(X_fit_pca50)

tsne_df = pd.DataFrame({
    "TSNE_1": X_fit_tsne[:, 0],
    "TSNE_2": X_fit_tsne[:, 1],
    "Cancer": y_fit.to_numpy()
})

plt.figure(figsize=(10,7))

for cancer_type in sorted(tsne_df["Cancer"].unique()):
    subset = tsne_df[tsne_df["Cancer"] == cancer_type]

    plt.scatter(
        subset["TSNE_1"],
        subset['TSNE_2'],
        label=cancer_type,
        alpha=0.7,
        s=25)

plt.title('t-SNE Visualization of Cancer Gene Expression')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.legend(title='Caner type')
plt.grid(True, alpha=0.3)
plt.show()

The t-SNE method reduces the dataset from high-dimension to two dimensions, allowing to visualize the cancer samples in a scatter plot, where each point represents one sample and the colors show the differenct cancer types. 

The t-SNE plot shows five distinct clusters in the gene expression data. This suggests that samples in the dataset have different gene-expression patterns and samples with similar patterns are likely to group together. 

Separation of the clusters indicate that the five cancer types have distinguishable gene-expression profils. 

Feature selection tests wheter using a smaller subset of genes can retain predicitve performance while reducing the number of features. 

In [ ]:
#Feature selection

#remove features with 0.0 variance
selector_var = VarianceThreshold(threshold=0.0)
remove_constants_X_fit = selector_var.fit_transform(X_fit_scaled)
remove_constants_X_val = selector_var.transform(X_val_scaled)

print("Features", X_fit_scaled.shape[1])
print("Features after removing constants:", remove_constants_X_fit.shape[1])

#genes to keep
k_values = [100, 500, 1000, 2000, 5000, 10000]

selection_results = []

for k in k_values: 
    selector_k = SelectKBest(score_func=f_classif, k=k)
    X_fit_scaled_selected = selector_k.fit_transform(remove_constants_X_fit, y_fit)
    X_val_scaled_selected = selector_k.transform(remove_constants_X_val)

    selection_logreg_k = LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=RANDOM_SEED
    )

    selection_logreg_k.fit(X_fit_scaled_selected, y_fit)

    pred_selection = selection_logreg_k.predict(X_val_scaled_selected)

    accuracy_selection = accuracy_score(y_val, pred_selection)
    f1_macro_selection = f1_score(y_val, pred_selection, average='macro')
    
    selection_results.append({
            "Selected genes": k,
            "Accuracy": accuracy_selection,
            "Macro f1-score": f1_macro_selection
        })
    
    print(f"\nSelected genes: {k}")
    print("Classification report:")
    print(classification_report(y_val, pred_selection))
    
results_selection_df = pd.DataFrame(selection_results)
print("Feature selection results:")
print(results_selection_df.round(3))

The genes that have no variance across all samples and therefore do not help distinguish between the cancer types, are removed.
The evaluatuon metric shows that feature selection can reduce the number of genes largerly while maintaining very high classification performance. 
The model does not need all 20530 original feautrues to achieve high predicitve performance. 
For 1000 genes both accuracy and Macro-F1 give the best overall balance and will be used for further model comparison